# Phase 4 · Model Training & Evaluation
### Huấn luyện, tinh chỉnh trên Validation, đánh giá một lần duy nhất trên Test

| 5W1H | Nội dung |
|---|---|
| **What** | Huấn luyện **15+ mô hình** chia thành 5 nhóm: (A) mô hình cũ tái lập, (B) tuyến tính và Naive Bayes trên TF-IDF, (C) cây và boosting (DT, RF, XGBoost, LightGBM, CatBoost), (D) Stacking và Soft Voting, (E) **một** baseline Deep Learning (MLP). |
| **Why** | Phase 1 chứng minh dữ liệu phi tuyến, Phase 3 cung cấp hai tập đặc trưng tối ưu. Phase này trả lời câu hỏi: *họ thuật toán nào, với cấu hình nào, cho hệ thống phát hiện phishing tốt nhất trên **domain chưa từng thấy**?* |
| **Where** | Đầu vào: `artifacts/data/*`, `artifacts/transformers/*`, `artifacts/reports/phase{2,3}_*.json`. Đầu ra: `artifacts/models/phishing_detector.joblib`, `artifacts/reports/phase4_*.csv`. |
| **How** | **Train** dùng để fit. **Validation** dùng để chọn siêu tham số, early stopping, trọng số ensemble và **ngưỡng quyết định**. **Test** được dùng **một lần duy nhất** ở §9, sau khi mọi lựa chọn đã chốt. |

In [ ]:
import sys
import json
import time
import warnings
from pathlib import Path


def _find_root(start: Path = Path.cwd()) -> Path:
    for p in [start, *start.parents]:
        if (p / 'data' / 'raw' / 'phishing_site_urls.csv').exists():
            return p
    raise FileNotFoundError('Không tìm thấy thư mục gốc dự án.')


ROOT = _find_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import optuna
import shap
import lightgbm as lgb
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from scipy import sparse

from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import ComplementNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import average_precision_score, roc_curve, precision_recall_curve, ConfusionMatrixDisplay

from src.phishing_ml.config import (DATA_DIR, TRANSFORMER_DIR, MODEL_DIR, REPORT_DIR, RANDOM_STATE, LABELS,
                                    ensure_dirs, seed_everything)
from src.phishing_ml.evaluation import (find_best_f1_threshold, threshold_for_fpr, compute_metrics, bootstrap_ci,
                                        plot_evaluation)
from src.phishing_ml.features import build_feature_frame, normalize_url, legacy_feature_frame

warnings.filterwarnings('ignore')
seed_everything()
ensure_dirs()
optuna.logging.set_verbosity(optuna.logging.WARNING)
sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.float_format', lambda v: f'{v:,.4f}')
pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 250)

N_TRIALS = 25            # số trial Optuna cho mỗi GBDT
SPLITS = ['train', 'val', 'test']

# ---------------------------------------------------------------- Nạp artifacts Phase 2 & 3
meta = json.loads((REPORT_DIR / 'phase2_metadata.json').read_text())
sel = json.loads((REPORT_DIR / 'phase3_feature_selection.json').read_text())
NUM_COLS, SPW = meta['num_cols'], meta['scale_pos_weight']
TREE_FEATURES, LINEAR_FEATURES, LEGACY_COLS = sel['tree_features'], sel['linear_features'], sel['legacy_features']

R = {k: pd.read_parquet(DATA_DIR / f'{k}_raw.parquet') for k in SPLITS}
S = {k: pd.read_parquet(DATA_DIR / f'{k}_scaled.parquet') for k in SPLITS}
L = {k: pd.read_parquet(DATA_DIR / f'{k}_legacy.parquet') for k in SPLITS}
selector = joblib.load(TRANSFORMER_DIR / 'tfidf_chi2_selector.joblib')
T = {k: selector.transform(sparse.load_npz(DATA_DIR / f'{k}_tfidf.npz')).tocsr() for k in SPLITS}
y = {k: R[k]['label'].to_numpy() for k in SPLITS}
groups_train = R['train']['registered_domain'].to_numpy()

X_tree = {k: R[k][NUM_COLS].astype(np.float32).assign(tld_te=S[k]['tld_te'].to_numpy())[TREE_FEATURES] for k in SPLITS}
X_lin = {k: S[k][LINEAR_FEATURES] for k in SPLITS}
CAT_FEATURES = [f for f in TREE_FEATURES if f != 'tld_te'] + ['tld']       # CatBoost dùng TLD dạng chuỗi gốc
X_cat = {k: R[k][CAT_FEATURES] for k in SPLITS}
SW_TRAIN = np.where(y['train'] == 1, meta['class_weights']['1'], meta['class_weights']['0'])

shapes = pd.DataFrame({k: {'Số mẫu': len(y[k]), '% phishing': y[k].mean() * 100, 'Legacy': L[k].shape[1],
                           'TREE': X_tree[k].shape[1], 'LINEAR/MLP': X_lin[k].shape[1], 'TF-IDF (χ²)': T[k].shape[1]}
                       for k in SPLITS}).T
display(shapes)

## 1. Lựa chọn Metrics (5-Whys)

**Bối cảnh vận hành:** mô hình chạy ở proxy hoặc email gateway, và mỗi ngày chấm hàng trăm nghìn URL. Có hai loại sai lầm với hậu quả **không đối xứng**:

| Lỗi | Ý nghĩa | Hậu quả thực tế |
|---|---|---|
| **False Negative** (bỏ sót phishing) | Link độc hại lọt tới người dùng | Mất thông tin đăng nhập, cài ransomware, xâm nhập lan rộng. Một FN có thể gây thiệt hại hàng triệu USD. |
| **False Positive** (chặn nhầm link sạch) | Người dùng không truy cập được site hợp lệ | Gián đoạn công việc, tạo ticket helpdesk, *alert fatigue* ở SOC. Người dùng mất niềm tin và tìm cách vượt qua bộ lọc. |

**Chuỗi 5-Whys dẫn tới bộ metrics:**
1. *Vì sao không dùng Accuracy làm thước đo chính?* Vì 78% URL là hợp lệ, nên một mô hình "luôn cho qua" đã đạt Accuracy 0.78 nhưng bỏ sót **100%** phishing.
2. *Vì sao cần cả Precision lẫn Recall?* **Recall** = tỉ lệ phishing bị bắt, đo trực tiếp rủi ro FN. **Precision** = tỉ lệ cảnh báo đúng, đo gánh nặng FP cho SOC. Mỗi chỉ số đơn lẻ đều bị "hack" được: chặn tất cả thì Recall = 1, chặn cực ít thì Precision = 1.
3. *Vì sao F1?* Trung bình điều hòa của P và R sẽ phạt nặng mô hình lệch hẳn về một phía. Ta dùng F1 để **chọn ngưỡng cân bằng** trên Validation.
4. *Vì sao ROC-AUC và **PR-AUC**?* Hai chỉ số này đo chất lượng **xếp hạng rủi ro** trên mọi ngưỡng, độc lập với ngưỡng đang chọn. Với dữ liệu mất cân bằng, **PR-AUC** nhạy hơn ROC-AUC vì nó không bị "pha loãng" bởi số lượng lớn True Negative. Vì vậy **PR-AUC là metric chính để chọn mô hình và siêu tham số**.
5. *Vì sao thêm **Recall@FPR≤1%**?* Đây là câu hỏi SOC thực sự đặt ra: *"Nếu chỉ cho phép chặn nhầm 1% lưu lượng sạch, ta bắt được bao nhiêu phần trăm phishing?"*. Chỉ số này gắn trực tiếp với chính sách **chặn tự động**.

| Metric | Vai trò trong notebook |
|---|---|
| **PR-AUC** | ⭐ Chọn mô hình, Optuna objective, early stopping |
| F1-Score | Chọn **ngưỡng** trên Validation (điểm vận hành cân bằng) |
| Precision / Recall / FPR / FNR | Diễn giải chi phí FP/FN tại ngưỡng đã chọn |
| ROC-AUC | Chất lượng xếp hạng tổng quát, so sánh với tài liệu |
| Recall@FPR≤1% | Hiệu năng ở chế độ chặn tự động |
| Accuracy, Brier | Tham khảo (Brier đo độ hiệu chỉnh xác suất) |

## 2. Giao thức đánh giá

- `register()` nhận một hàm `predict(split) → xác suất`. Hàm này chỉ chạy trên **Validation** để: (1) chọn ngưỡng tối đa F1 (trừ mô hình cũ, vốn dùng ngưỡng 0.5 như code gốc), và (2) ghi metrics validation.
- Dự đoán trên Test **chưa được tính ở bước này**. Ta chỉ gọi `predict('test')` **một lần** ở §9, sau khi bảng xếp hạng Validation đã chốt.

In [ ]:
REGISTRY, VAL_RESULTS = {}, {}
G_OLD, G_LIN, G_TREE, G_ENS, G_DL = ('A. Mô hình cũ (baseline)', 'B. Tuyến tính / Naive Bayes',
                                     'C. Cây & Boosting', 'D. Ensemble', 'E. Baseline Deep Learning')


def register(name, group, predict_fn, fixed_threshold=None, train_time=np.nan, note=''):
    p_val = np.asarray(predict_fn('val'), dtype=float)
    thr = fixed_threshold if fixed_threshold is not None else find_best_f1_threshold(y['val'], p_val)
    m = compute_metrics(y['val'], p_val, thr)
    REGISTRY[name] = {'group': group, 'predict': predict_fn, 'threshold': thr, 'p_val': p_val}
    VAL_RESULTS[name] = {'Nhóm': group, 'Train time (s)': train_time, 'Ghi chú': note, **m}
    print(f'[{group}] {name:<42} val PR-AUC = {m["PR-AUC"]:.4f} | F1 = {m["F1-Score"]:.4f} '
          f'@ thr {thr:.3f} | train {train_time:,.1f}s')


def timed(fn):
    t0 = time.time()
    out = fn()
    return out, time.time() - t0

## 3. Nhóm A: Tái lập mô hình cũ (Baseline)

Ta tái lập chính xác 3 mô hình của notebook 05: **20 đặc trưng cũ, siêu tham số cũ, ngưỡng 0.5**. Điểm khác duy nhất là chúng được huấn luyện và đánh giá trên **cùng split theo domain** với mô hình mới, để phép so sánh công bằng.

In [ ]:
old_lr, t = timed(lambda: LogisticRegression(max_iter=1000).fit(L['train'], y['train']))
register('Old · Logistic Regression (20 feat)', G_OLD, lambda s: old_lr.predict_proba(L[s])[:, 1], 0.5, t)

old_rf, t = timed(lambda: RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
                  .fit(L['train'], y['train']))
register('Old · Random Forest (20 feat)', G_OLD, lambda s: old_rf.predict_proba(L[s])[:, 1], 0.5, t)

old_xgb, t = timed(lambda: XGBClassifier(n_estimators=300, max_depth=10, learning_rate=0.05, subsample=0.9,
                                         colsample_bytree=0.8, eval_metric='logloss', random_state=RANDOM_STATE,
                                         n_jobs=-1).fit(L['train'], y['train']))
register('Old · XGBoost tuned NB05 (20 feat)', G_OLD, lambda s: old_xgb.predict_proba(L[s])[:, 1], 0.5, t)

## 4. Nhóm B: Mô hình tuyến tính và Naive Bayes

Có hai cách dùng mô hình tuyến tính:
- **Trên TF-IDF n-gram (χ²-selected)**: không gian nhiều chiều làm dữ liệu gần tách được tuyến tính (định lý Cover, Phase 1 §6). Đây là nơi mô hình tuyến tính mạnh nhất.
- **Trên đặc trưng thủ công `LINEAR_FEATURES`** (đã Winsorize, Yeo-Johnson, scale, lọc VIF): mô hình đối chứng trực tiếp với cây trên cùng thông tin, nhằm kiểm chứng lại kết luận phi tuyến của Phase 1.

Siêu tham số được chọn theo **PR-AUC trên Validation**.

In [ ]:
# 4.1 Logistic Regression + TF-IDF: chọn C
lr_grid = {}
for C in [2.0, 8.0, 32.0]:
    m, t = timed(lambda: LogisticRegression(C=C, solver='liblinear', class_weight='balanced', max_iter=500)
                 .fit(T['train'], y['train']))
    lr_grid[C] = (average_precision_score(y['val'], m.predict_proba(T['val'])[:, 1]), m, t)
    print(f'  LR TF-IDF C = {C:<5} val PR-AUC = {lr_grid[C][0]:.4f} ({t:.1f}s)')
BEST_C = max(lr_grid, key=lambda c: lr_grid[c][0])
lr_tfidf = lr_grid[BEST_C][1]
register('LR + TF-IDF', G_LIN, lambda s: lr_tfidf.predict_proba(T[s])[:, 1], train_time=lr_grid[BEST_C][2],
         note=f'C={BEST_C}')

# 4.2 Linear SVM + TF-IDF: chọn C trên decision_function, sau đó hiệu chỉnh xác suất (Platt, 3-fold)
svm_grid = {}
for C in [0.05, 0.2]:
    m, t = timed(lambda: LinearSVC(C=C, class_weight='balanced').fit(T['train'], y['train']))
    svm_grid[C] = average_precision_score(y['val'], m.decision_function(T['val']))
    print(f'  LinearSVC C = {C:<5} val PR-AUC = {svm_grid[C]:.4f} ({t:.1f}s)')
BEST_SVM_C = max(svm_grid, key=svm_grid.get)
svm_cal, t = timed(lambda: CalibratedClassifierCV(LinearSVC(C=BEST_SVM_C, class_weight='balanced'), method='sigmoid',
                                                  cv=3).fit(T['train'], y['train']))
register('Linear SVM (calibrated) + TF-IDF', G_LIN, lambda s: svm_cal.predict_proba(T[s])[:, 1], train_time=t,
         note=f'C={BEST_SVM_C}')

# 4.3 Complement Naive Bayes + TF-IDF: chọn alpha
nb_grid = {}
for a in [0.01, 0.1, 0.5]:
    m, t = timed(lambda: ComplementNB(alpha=a).fit(T['train'], y['train']))
    nb_grid[a] = (average_precision_score(y['val'], m.predict_proba(T['val'])[:, 1]), m, t)
    print(f'  ComplementNB alpha = {a:<5} val PR-AUC = {nb_grid[a][0]:.4f}')
BEST_ALPHA = max(nb_grid, key=lambda a: nb_grid[a][0])
cnb = nb_grid[BEST_ALPHA][1]
register('Complement NB + TF-IDF', G_LIN, lambda s: cnb.predict_proba(T[s])[:, 1], train_time=nb_grid[BEST_ALPHA][2],
         note=f'alpha={BEST_ALPHA}')

# 4.4 Logistic Regression trên đặc trưng thủ công (đối chứng tuyến tính vs phi tuyến)
lr_hand, t = timed(lambda: LogisticRegression(max_iter=3000, class_weight='balanced').fit(X_lin['train'], y['train']))
register('LR + hand-crafted (LINEAR_FEATURES)', G_LIN, lambda s: lr_hand.predict_proba(X_lin[s])[:, 1], train_time=t)

## 5. Nhóm C: Cây quyết định và Boosting (trên `TREE_FEATURES`, giá trị gốc)

| Mô hình | Chiến lược tinh chỉnh trên Validation |
|---|---|
| Decision Tree | Lưới `max_depth` |
| Random Forest | Lưới `min_samples_leaf` (200 cây, `class_weight='balanced_subsample'`) |
| XGBoost | **Optuna TPE** (25 trials) trên 8 siêu tham số, objective = val PR-AUC, early stopping |
| LightGBM | **Optuna TPE** (25 trials), leaf-wise growth, early stopping |
| CatBoost | Lưới `depth`, **TLD dạng categorical gốc** (ordered target statistics), early stopping |

Cả năm mô hình đều xử lý mất cân bằng bằng **trọng số lớp** (`scale_pos_weight` hoặc `class_weight`), đúng quyết định ở Phase 2.

In [ ]:
# 5.1 Decision Tree
dt_grid = {}
for d in [8, 12, 16, 22]:
    m, t = timed(lambda: DecisionTreeClassifier(max_depth=d, min_samples_leaf=20, class_weight='balanced',
                                                random_state=RANDOM_STATE).fit(X_tree['train'], y['train']))
    dt_grid[d] = (average_precision_score(y['val'], m.predict_proba(X_tree['val'])[:, 1]), m, t)
    print(f'  DecisionTree max_depth = {d:<3} val PR-AUC = {dt_grid[d][0]:.4f}')
BEST_D = max(dt_grid, key=lambda d: dt_grid[d][0])
dt = dt_grid[BEST_D][1]
register('Decision Tree', G_TREE, lambda s: dt.predict_proba(X_tree[s])[:, 1], train_time=dt_grid[BEST_D][2],
         note=f'max_depth={BEST_D}')

# 5.2 Random Forest
rf_grid = {}
for leaf in [2, 8]:
    m, t = timed(lambda: RandomForestClassifier(n_estimators=200, min_samples_leaf=leaf, max_features='sqrt',
                                                class_weight='balanced_subsample', n_jobs=-1,
                                                random_state=RANDOM_STATE).fit(X_tree['train'], y['train']))
    rf_grid[leaf] = (average_precision_score(y['val'], m.predict_proba(X_tree['val'])[:, 1]), m, t)
    print(f'  RandomForest min_samples_leaf = {leaf:<3} val PR-AUC = {rf_grid[leaf][0]:.4f} ({t:.1f}s)')
BEST_LEAF = max(rf_grid, key=lambda k: rf_grid[k][0])
rf = rf_grid[BEST_LEAF][1]
rf_grid = {BEST_LEAF: rf_grid[BEST_LEAF]}      # giải phóng bộ nhớ của cấu hình thua
register('Random Forest', G_TREE, lambda s: rf.predict_proba(X_tree[s])[:, 1], train_time=rf_grid[BEST_LEAF][2],
         note=f'min_samples_leaf={BEST_LEAF}')

In [ ]:
# 5.3 XGBoost + Optuna
def make_xgb(params, n_estimators=3000):
    return XGBClassifier(n_estimators=n_estimators, tree_method='hist', eval_metric='aucpr', early_stopping_rounds=100,
                         scale_pos_weight=SPW, random_state=RANDOM_STATE, n_jobs=-1, **params)


def xgb_objective(trial):
    params = {'learning_rate': trial.suggest_float('learning_rate', 0.02, 0.3, log=True),
              'max_depth': trial.suggest_int('max_depth', 4, 12),
              'min_child_weight': trial.suggest_float('min_child_weight', 1.0, 30.0, log=True),
              'subsample': trial.suggest_float('subsample', 0.6, 1.0),
              'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
              'gamma': trial.suggest_float('gamma', 0.0, 5.0),
              'reg_lambda': trial.suggest_float('reg_lambda', 1e-2, 20.0, log=True),
              'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 5.0, log=True)}
    m = make_xgb(params).fit(X_tree['train'], y['train'], eval_set=[(X_tree['val'], y['val'])], verbose=False)
    return average_precision_score(y['val'], m.predict_proba(X_tree['val'])[:, 1])


t0 = time.time()
xgb_study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
xgb_study.optimize(xgb_objective, n_trials=N_TRIALS)
xgb, t = timed(lambda: make_xgb(xgb_study.best_params).fit(X_tree['train'], y['train'],
                                                          eval_set=[(X_tree['val'], y['val'])], verbose=False))
print(f'XGBoost Optuna: {N_TRIALS} trials trong {(time.time() - t0) / 60:.1f} phút | best_iteration = {xgb.best_iteration}')
register('XGBoost + Optuna', G_TREE, lambda s: xgb.predict_proba(X_tree[s])[:, 1], train_time=t,
         note=f'{xgb.best_iteration + 1} cây')

# 5.4 LightGBM + Optuna
def make_lgbm(params, n_estimators=3000):
    return lgb.LGBMClassifier(n_estimators=n_estimators, scale_pos_weight=SPW, subsample_freq=1,
                              random_state=RANDOM_STATE, n_jobs=-1, verbose=-1, **params)


def lgbm_fit(params, Xtr, Xva):
    return make_lgbm(params).fit(Xtr, y['train'], eval_set=[(Xva, y['val'])], eval_metric='average_precision',
                                 callbacks=[lgb.early_stopping(100, verbose=False)])


def lgbm_objective(trial):
    params = {'learning_rate': trial.suggest_float('learning_rate', 0.02, 0.2, log=True),
              'num_leaves': trial.suggest_int('num_leaves', 31, 255, log=True),
              'min_child_samples': trial.suggest_int('min_child_samples', 10, 200, log=True),
              'subsample': trial.suggest_float('subsample', 0.6, 1.0),
              'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
              'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
              'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 5.0, log=True)}
    m = lgbm_fit(params, X_tree['train'], X_tree['val'])
    return average_precision_score(y['val'], m.predict_proba(X_tree['val'])[:, 1])


t0 = time.time()
lgbm_study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
lgbm_study.optimize(lgbm_objective, n_trials=N_TRIALS)
lgbm, t = timed(lambda: lgbm_fit(lgbm_study.best_params, X_tree['train'], X_tree['val']))
print(f'LightGBM Optuna: {N_TRIALS} trials trong {(time.time() - t0) / 60:.1f} phút | best_iteration = {lgbm.best_iteration_}')
register('LightGBM + Optuna', G_TREE, lambda s: lgbm.predict_proba(X_tree[s])[:, 1], train_time=t,
         note=f'{lgbm.best_iteration_} cây')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(17, 4.5))
for ax, (name, study) in zip(axes, [('XGBoost', xgb_study), ('LightGBM', lgbm_study)]):
    vals = np.array([tr.value for tr in study.trials])
    ax.scatter(range(len(vals)), vals, alpha=0.7, label='PR-AUC từng trial')
    ax.plot(np.maximum.accumulate(vals), c='#e34a33', lw=2, label='Tốt nhất tới hiện tại')
    ax.set(title=f'Optuna: {name}', xlabel='Trial', ylabel='Validation PR-AUC')
    ax.legend()
plt.tight_layout()
plt.show()
display(pd.DataFrame({'XGBoost': pd.Series(xgb_study.best_params), 'LightGBM': pd.Series(lgbm_study.best_params)}))

In [ ]:
# 5.5 CatBoost: TLD dạng categorical gốc
cat_grid = {}
for depth in [6, 8]:
    m, t = timed(lambda: CatBoostClassifier(iterations=2500, learning_rate=0.08, depth=depth, eval_metric='PRAUC',
                                            scale_pos_weight=SPW, od_type='Iter', od_wait=100, random_seed=RANDOM_STATE,
                                            thread_count=-1, verbose=0, cat_features=['tld'])
                 .fit(X_cat['train'], y['train'], eval_set=(X_cat['val'], y['val']), use_best_model=True))
    cat_grid[depth] = (average_precision_score(y['val'], m.predict_proba(X_cat['val'])[:, 1]), m, t)
    print(f'  CatBoost depth = {depth} val PR-AUC = {cat_grid[depth][0]:.4f} | {m.get_best_iteration() + 1} cây ({t:.1f}s)')
BEST_DEPTH = max(cat_grid, key=lambda d: cat_grid[d][0])
catb = cat_grid[BEST_DEPTH][1]
register('CatBoost (native categorical TLD)', G_TREE, lambda s: catb.predict_proba(X_cat[s])[:, 1],
         train_time=cat_grid[BEST_DEPTH][2], note=f'depth={BEST_DEPTH}')

## 6. Nhóm D: Ensemble (Stacking và Soft Voting)

### 6.1. Stacking: n-gram → meta-GBDT
- **Level-0:** `LR + TF-IDF` và `Complement NB + TF-IDF` tạo điểm rủi ro từ chuỗi ký tự. Trên **Train**, điểm này là **out-of-fold**: 5-fold `StratifiedGroupKFold` theo domain, nên mỗi URL được chấm bởi mô hình **chưa thấy domain của nó**. Nếu dùng dự đoán in-sample, meta-model sẽ học rằng "điểm LR luôn đúng" (LR gần như thuộc lòng train) và sẽ sụp đổ khi gặp dữ liệu mới.
- **Level-1:** LightGBM (dùng siêu tham số Optuna ở §5.4) trên `TREE_FEATURES` + 2 điểm level-0. Meta-model học **khi nào tin chuỗi ký tự, khi nào tin tín hiệu bảo mật** (brand, TLD, entropy).

In [ ]:
t0 = time.time()
oof = np.zeros((len(y['train']), 2), dtype=np.float32)
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
for k, (fit_i, oof_i) in enumerate(sgkf.split(np.zeros(len(y['train'])), y['train'], groups_train), 1):
    m_lr = LogisticRegression(C=BEST_C, solver='liblinear', class_weight='balanced', max_iter=500)
    m_lr.fit(T['train'][fit_i], y['train'][fit_i])
    m_nb = ComplementNB(alpha=BEST_ALPHA).fit(T['train'][fit_i], y['train'][fit_i])
    oof[oof_i, 0] = m_lr.predict_proba(T['train'][oof_i])[:, 1]
    oof[oof_i, 1] = m_nb.predict_proba(T['train'][oof_i])[:, 1]
    print(f'  Fold {k}: OOF PR-AUC  LR = {average_precision_score(y["train"][oof_i], oof[oof_i, 0]):.4f} | '
          f'NB = {average_precision_score(y["train"][oof_i], oof[oof_i, 1]):.4f}')
print(f'OOF xong trong {time.time() - t0:.1f}s')

LEVEL0 = ['lr_tfidf_score', 'nb_tfidf_score']


def stack_frame(split):
    if split == 'train':
        scores = oof
    else:
        scores = np.column_stack([lr_tfidf.predict_proba(T[split])[:, 1], cnb.predict_proba(T[split])[:, 1]])
    return X_tree[split].assign(**{c: scores[:, i].astype(np.float32) for i, c in enumerate(LEVEL0)})


X_stack = {s: stack_frame(s) for s in ['train', 'val']}
stack_meta, t = timed(lambda: lgbm_fit(lgbm_study.best_params, X_stack['train'], X_stack['val']))
register('Stacking (TF-IDF LR+NB → LightGBM)', G_ENS, lambda s: stack_meta.predict_proba(stack_frame(s))[:, 1],
         train_time=t, note=f'meta {stack_meta.best_iteration_} cây')

### 6.2. Soft Voting: trung bình có trọng số
p = w·p(LR TF-IDF) + (1 − w)·p(GBDT tốt nhất trên đặc trưng thủ công). Trọng số w được chọn theo PR-AUC trên Validation.

In [ ]:
gbdt_names = ['XGBoost + Optuna', 'LightGBM + Optuna', 'CatBoost (native categorical TLD)']
best_gbdt = max(gbdt_names, key=lambda n: VAL_RESULTS[n]['PR-AUC'])
p_lr_val, p_gb_val = REGISTRY['LR + TF-IDF']['p_val'], REGISTRY[best_gbdt]['p_val']
weights = np.round(np.arange(0, 1.0001, 0.05), 2)
w_scores = [average_precision_score(y['val'], w * p_lr_val + (1 - w) * p_gb_val) for w in weights]
BEST_W = float(weights[int(np.argmax(w_scores))])
gb_predict = REGISTRY[best_gbdt]['predict']

fig, ax = plt.subplots(figsize=(9, 3.8))
ax.plot(weights, w_scores, marker='o')
ax.axvline(BEST_W, ls='--', c='#e34a33', label=f'w* = {BEST_W}')
ax.set(xlabel=f'w (trọng số LR TF-IDF; phần còn lại cho {best_gbdt})', ylabel='Validation PR-AUC',
       title='Chọn trọng số Soft Voting trên Validation')
ax.legend()
plt.tight_layout()
plt.show()
register(f'Soft Voting (LR TF-IDF + {best_gbdt.split(" ")[0]})', G_ENS,
         lambda s: BEST_W * lr_tfidf.predict_proba(T[s])[:, 1] + (1 - BEST_W) * gb_predict(s), train_time=0.0,
         note=f'w={BEST_W} (không huấn luyện thêm)')

## 7. Nhóm E: Baseline Deep Learning (MLPClassifier)

**Vì sao chỉ một MLP?** Ràng buộc dự án là thuần ML truyền thống, và DL chỉ làm **mốc đối chiếu**. MLP là mạng nơ-ron truyền thẳng cơ bản nhất: 3 lớp ẩn (256-128-64, ReLU), Adam, early stopping. Mạng dùng `LINEAR_FEATURES` (đã chuẩn hóa, giống LR) vì mạng nơ-ron hội tụ kém khi đầu vào lệch thang đo, và dùng `sample_weight` theo class weights để cân bằng lớp.

In [ ]:
mlp, t = timed(lambda: MLPClassifier(hidden_layer_sizes=(256, 128, 64), activation='relu', solver='adam', alpha=1e-4,
                                     batch_size=1024, learning_rate_init=1e-3, max_iter=60, early_stopping=True,
                                     validation_fraction=0.1, n_iter_no_change=6, random_state=RANDOM_STATE)
               .fit(X_lin['train'], y['train'], sample_weight=SW_TRAIN))
print(f'MLP: {mlp.n_iter_} epoch, best internal val score = {mlp.best_validation_score_:.4f}')
register('MLP (256-128-64)', G_DL, lambda s: mlp.predict_proba(X_lin[s])[:, 1], train_time=t,
         note=f'{mlp.n_iter_} epoch')

fig, ax = plt.subplots(figsize=(9, 3.8))
ax.plot(mlp.loss_curve_, label='Training loss')
ax2 = ax.twinx()
ax2.plot(mlp.validation_scores_, c='#e34a33', label='Internal val accuracy')
ax.set(xlabel='Epoch', ylabel='Loss', title='MLP: đường cong học')
fig.legend(loc='upper right', bbox_to_anchor=(0.88, 0.85))
plt.tight_layout()
plt.show()

## 8. Bảng xếp hạng trên VALIDATION (cơ sở để chốt mô hình)

Bảng này được dùng để chọn mô hình tốt nhất trong từng nhóm. Đến thời điểm này **Test vẫn chưa được chạm tới**.

In [ ]:
SHOW_COLS = ['Nhóm', 'PR-AUC', 'ROC-AUC', 'F1-Score', 'Precision', 'Recall', 'Recall@FPR≤1%', 'Threshold',
             'Train time (s)', 'Ghi chú']
val_df = pd.DataFrame(VAL_RESULTS).T[SHOW_COLS].sort_values('PR-AUC', ascending=False)
val_df.to_csv(REPORT_DIR / 'phase4_validation_leaderboard.csv')
display(val_df.style.format({c: '{:.4f}' for c in SHOW_COLS[1:8]} | {'Train time (s)': '{:,.1f}'})
        .background_gradient(subset=['PR-AUC'], cmap='Greens'))

BEST_BY_GROUP = val_df.reset_index().groupby('Nhóm', sort=True).first()['index'].to_dict()
ML_GROUPS = [G_LIN, G_TREE, G_ENS]
BEST_ML = val_df[val_df['Nhóm'].isin(ML_GROUPS)].index[0]
BEST_OLD, BEST_DL = BEST_BY_GROUP[G_OLD], BEST_BY_GROUP[G_DL]
print(f'Mô hình cũ tốt nhất : {BEST_OLD}')
print(f'ML cải tiến tốt nhất: {BEST_ML}')
print(f'Baseline DL         : {BEST_DL}')

## 9. Đánh giá CUỐI CÙNG trên TEST (chạy một lần duy nhất)

Mọi mô hình dùng **ngưỡng đã chốt trên Validation**. Test gồm khoảng 100 nghìn URL thuộc các **domain không xuất hiện** trong Train hay Validation.

In [ ]:
TEST_RESULTS, P_TEST = {}, {}
for name, reg in REGISTRY.items():
    P_TEST[name] = np.asarray(reg['predict']('test'), dtype=float)          # ← lần duy nhất chạm vào test
    TEST_RESULTS[name] = {'Nhóm': reg['group'], **compute_metrics(y['test'], P_TEST[name], reg['threshold'])}

METRIC_COLS = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC', 'PR-AUC', 'Recall@FPR≤1%', 'FPR', 'FNR',
               'Brier', 'Threshold', 'FP', 'FN']
test_df = pd.DataFrame(TEST_RESULTS).T[['Nhóm'] + METRIC_COLS]
test_df[METRIC_COLS] = test_df[METRIC_COLS].astype(float)
test_df = test_df.sort_values(['Nhóm', 'PR-AUC'], ascending=[True, False])
test_df.to_csv(REPORT_DIR / 'phase4_test_results.csv')
pd.DataFrame(P_TEST).assign(label=y['test']).to_parquet(REPORT_DIR / 'phase4_test_probas.parquet', index=False)

higher = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC', 'PR-AUC', 'Recall@FPR≤1%']
display(test_df.style
        .format({c: '{:.4f}' for c in METRIC_COLS if c not in ('FP', 'FN')} | {'FP': '{:,.0f}', 'FN': '{:,.0f}'})
        .highlight_max(subset=higher, color='#c7e9c0').highlight_min(subset=['FPR', 'FNR', 'Brier', 'FP', 'FN'],
                                                                     color='#c7e9c0')
        .set_caption('Kết quả trên TEST (split theo domain); ngưỡng chọn trên Validation, mô hình cũ dùng 0.5'))

### 9.1. ROC và Precision–Recall của tất cả mô hình

Ba mô hình đại diện được tô đậm: **mô hình cũ tốt nhất**, **ML cải tiến tốt nhất** và **baseline DL**. Panel giữa phóng to vùng **FPR ≤ 5%**, là vùng vận hành thực tế của hệ thống chặn tự động.

In [ ]:
KEY = {BEST_OLD: '#7f7f7f', BEST_ML: '#d62728', BEST_DL: '#1f77b4'}
fig, axes = plt.subplots(1, 3, figsize=(24, 7))
for name, p in P_TEST.items():
    fpr, tpr, _ = roc_curve(y['test'], p)
    prec, rec, _ = precision_recall_curve(y['test'], p)
    key = name in KEY
    style = dict(lw=2.8 if key else 0.9, alpha=1 if key else 0.45, color=KEY.get(name),
                 ls='-' if key else ('--' if REGISTRY[name]['group'] == G_OLD else '-'))
    for ax in axes[:2]:
        ax.plot(fpr, tpr, label=f'{name} ({TEST_RESULTS[name]["ROC-AUC"]:.3f})' if key or ax is axes[0] else None, **style)
    axes[2].plot(rec, prec, label=f'{name} ({TEST_RESULTS[name]["PR-AUC"]:.3f})', **style)
axes[0].plot([0, 1], [0, 1], 'k:', lw=1)
axes[0].set(title='ROC (test)', xlabel='FPR', ylabel='TPR')
axes[1].set(title='ROC phóng to: FPR ≤ 5%', xlabel='FPR', ylabel='TPR', xlim=(0, 0.05), ylim=(0.2, 1.0))
axes[1].axvline(0.01, c='grey', ls=':', lw=1)
axes[2].axhline(y['test'].mean(), c='k', ls=':', lw=1)
axes[2].set(title='Precision–Recall (test)', xlabel='Recall', ylabel='Precision', ylim=(0.2, 1.01))
axes[0].legend(fontsize=7.5, loc='lower right')
axes[1].legend(fontsize=9, loc='lower right')
axes[2].legend(fontsize=7.5, loc='lower left')
plt.tight_layout()
plt.show()

### 9.2. Confusion Matrix: Mô hình cũ vs. ML cải tiến vs. Baseline DL

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(21, 6))
for ax, name in zip(axes, [BEST_OLD, BEST_ML, BEST_DL]):
    thr = REGISTRY[name]['threshold']
    ConfusionMatrixDisplay.from_predictions(y['test'], (P_TEST[name] >= thr).astype(int), display_labels=LABELS,
                                            cmap='Blues', values_format=',', ax=ax, colorbar=False)
    r = TEST_RESULTS[name]
    ax.set_title(f'{name}\nngưỡng {thr:.3f} | FP = {r["FP"]:,} | FN = {r["FN"]:,}', fontsize=11, fontweight='bold')
    ax.grid(False)
plt.tight_layout()
plt.show()
plot_evaluation(BEST_ML, y['test'], P_TEST[BEST_ML], REGISTRY[BEST_ML]['threshold'])

### 9.3. Độ tin cậy thống kê: khoảng tin cậy 95% bằng bootstrap

Một con số đơn lẻ trên Test chưa đủ để kết luận "A tốt hơn B". Ta lấy mẫu lại có hoàn lại 200 lần trên tập Test và tính khoảng tin cậy percentile 95%. Nếu hai khoảng **không giao nhau** thì chênh lệch có ý nghĩa thống kê.

In [ ]:
ci_rows = []
for name in [BEST_OLD, BEST_ML, BEST_DL]:
    ci = bootstrap_ci(y['test'], P_TEST[name], REGISTRY[name]['threshold'], n_boot=200)
    ci_rows.append({'Mô hình': name, **{f'{k} [95% CI]': f'{TEST_RESULTS[name][k]:.4f} [{lo:.4f}, {hi:.4f}]'
                                         for k, (lo, hi) in ci.items()}})
ci_df = pd.DataFrame(ci_rows).set_index('Mô hình')
ci_df.to_csv(REPORT_DIR / 'phase4_bootstrap_ci.csv')
display(ci_df)

### 9.4. Chính sách vận hành hai ngưỡng (chọn trên Validation, kiểm chứng trên Test)

Ngưỡng F1 là một điểm cân bằng phục vụ học thuật. SOC cần **chính sách**: 🔴 *chặn tự động* khi rất chắc chắn (FPR ≤ 0.3%), 🟡 *cảnh báo/sandbox* ở vùng xám (FPR ≤ 3%), 🟢 cho qua phần còn lại.

In [ ]:
p_val_best, p_test_best = REGISTRY[BEST_ML]['p_val'], P_TEST[BEST_ML]
T_BLOCK = threshold_for_fpr(y['val'], p_val_best, 0.003)
T_REVIEW = threshold_for_fpr(y['val'], p_val_best, 0.03)
zone = np.select([p_test_best >= T_BLOCK, p_test_best >= T_REVIEW], ['🔴 Chặn tự động', '🟡 Cảnh báo / Sandbox'],
                 '🟢 Cho phép')
policy = (pd.DataFrame({'Vùng': zone, 'y': y['test']}).groupby('Vùng')['y']
          .agg(**{'Số URL': 'size', 'Số phishing': 'sum'})
          .assign(**{'% phishing trong vùng': lambda d: d['Số phishing'] / d['Số URL'] * 100,
                     '% tổng phishing bị bắt': lambda d: d['Số phishing'] / y['test'].sum() * 100,
                     '% URL sạch rơi vào vùng': lambda d: (d['Số URL'] - d['Số phishing']) / (y['test'] == 0).sum() * 100}))
print(f'{BEST_ML}: T_block = {T_BLOCK:.4f} (FPR_val ≤ 0.3%) | T_review = {T_REVIEW:.4f} (FPR_val ≤ 3%)')
display(policy)

## 10. Khả năng giải thích: Feature Importance và SHAP

### 10.1. SHAP values của mô hình tốt nhất trên đặc trưng bảng

SHAP (TreeExplainer) phân rã **mỗi dự đoán** thành tổng đóng góp của từng đặc trưng, dựa trên giá trị Shapley của lý thuyết trò chơi. Biểu đồ beeswarm cho biết cả **độ lớn** lẫn **chiều** tác động. Ví dụ: chấm đỏ (giá trị cao) nằm bên phải nghĩa là giá trị cao **đẩy về phía phishing**. Các giá trị được tính trên 3,000 URL của tập Test. Đây là bước **giải thích**, không phải bước lựa chọn mô hình, nên không gây rò rỉ.

In [ ]:
tree_like = {'Stacking (TF-IDF LR+NB → LightGBM)': (stack_meta, lambda s: stack_frame(s)),
             'LightGBM + Optuna': (lgbm, lambda s: X_tree[s]), 'XGBoost + Optuna': (xgb, lambda s: X_tree[s])}
SHAP_MODEL = BEST_ML if BEST_ML in tree_like else max(tree_like, key=lambda n: VAL_RESULTS[n]['PR-AUC'])
shap_model, shap_X = tree_like[SHAP_MODEL]
Xs_shap = shap_X('test').sample(3000, random_state=RANDOM_STATE)
explainer = shap.TreeExplainer(shap_model)
sv = explainer.shap_values(Xs_shap)
sv = sv[1] if isinstance(sv, list) else sv
print(f'SHAP cho: {SHAP_MODEL}')

shap.summary_plot(sv, Xs_shap, max_display=20, show=False, plot_size=(11, 9))
plt.title(f'SHAP beeswarm: {SHAP_MODEL}', fontweight='bold')
plt.tight_layout()
plt.show()

shap_imp = pd.Series(np.abs(sv).mean(axis=0), index=Xs_shap.columns).sort_values(ascending=False)
if hasattr(shap_model, 'booster_'):                                    # LightGBM
    gain_imp = pd.Series(shap_model.booster_.feature_importance('gain'), index=Xs_shap.columns)
else:                                                                  # XGBoost
    gain_imp = pd.Series(shap_model.get_booster().get_score(importance_type='total_gain')).reindex(Xs_shap.columns)
gain_imp = (gain_imp.fillna(0) / gain_imp.sum()).sort_values(ascending=False)
fig, axes = plt.subplots(1, 2, figsize=(18, 8))
shap_imp.head(20)[::-1].plot(kind='barh', ax=axes[0], color='#7570b3')
axes[0].set(title='Mean |SHAP| (test sample)', xlabel='mean |SHAP value| (log-odds)')
gain_imp.head(20)[::-1].plot(kind='barh', ax=axes[1], color='#1b9e77')
axes[1].set(title='Tỉ trọng total gain', xlabel='Tỉ trọng')
plt.suptitle(f'Feature Importance: {SHAP_MODEL}', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

### 10.2. Giải thích nhánh n-gram: hệ số Logistic Regression

In [ ]:
char_vec = joblib.load(TRANSFORMER_DIR / 'tfidf_char.joblib')
word_vec = joblib.load(TRANSFORMER_DIR / 'tfidf_word.joblib')
vocab = np.concatenate([char_vec.get_feature_names_out().astype(str),
                        np.char.add('tok:', word_vec.get_feature_names_out().astype(str))])[selector.get_support()]
coef = lr_tfidf.coef_.ravel()
order = np.argsort(coef)
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
axes[0].barh([f"'{vocab[i]}'" for i in order[-20:]], coef[order[-20:]], color='#e34a33')
axes[0].set_title('Top 20 n-gram → PHISHING', fontweight='bold')
axes[1].barh([f"'{vocab[i]}'" for i in order[:20][::-1]], coef[order[:20][::-1]], color='#2b8cbe')
axes[1].set_title('Top 20 n-gram → HỢP LỆ', fontweight='bold')
for ax in axes:
    ax.set_xlabel('Hệ số LR')
plt.tight_layout()
plt.show()

## 11. Bảng tổng hợp: Mô hình cũ vs. ML cải tiến vs. Baseline DL

In [ ]:
summary = test_df.loc[[BEST_OLD, BEST_ML, BEST_DL], ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC', 'PR-AUC',
                                                      'Recall@FPR≤1%', 'FPR', 'FNR', 'FP', 'FN']]
summary.insert(0, 'Vai trò', ['Mô hình cũ (baseline)', 'ML cải tiến (tốt nhất)', 'Baseline Deep Learning'])
delta = summary.loc[BEST_ML, 'Accuracy':'FNR'].astype(float) - summary.loc[BEST_OLD, 'Accuracy':'FNR'].astype(float)
delta_dl = summary.loc[BEST_ML, 'Accuracy':'FNR'].astype(float) - summary.loc[BEST_DL, 'Accuracy':'FNR'].astype(float)
summary.loc['Δ (ML cải tiến − Mô hình cũ)'] = ['—', *delta.values,
                                                summary.loc[BEST_ML, 'FP'] - summary.loc[BEST_OLD, 'FP'],
                                                summary.loc[BEST_ML, 'FN'] - summary.loc[BEST_OLD, 'FN']]
summary.loc['Δ (ML cải tiến − Baseline DL)'] = ['—', *delta_dl.values,
                                                 summary.loc[BEST_ML, 'FP'] - summary.loc[BEST_DL, 'FP'],
                                                 summary.loc[BEST_ML, 'FN'] - summary.loc[BEST_DL, 'FN']]
summary.to_csv(REPORT_DIR / 'phase4_summary.csv')
display(summary.style.format({c: '{:.4f}' for c in summary.columns[1:-2]} |
                             {'FP': '{:,.0f}', 'FN': '{:,.0f}'}))
print(f'FN giảm {1 - summary.loc[BEST_ML, "FN"] / summary.loc[BEST_OLD, "FN"]:.1%} | '
      f'FP giảm {1 - summary.loc[BEST_ML, "FP"] / summary.loc[BEST_OLD, "FP"]:.1%} so với mô hình cũ')

fig, ax = plt.subplots(figsize=(15, 5.5))
plot_cols = ['Precision', 'Recall', 'F1-Score', 'ROC-AUC', 'PR-AUC', 'Recall@FPR≤1%']
summary.iloc[:3][plot_cols].astype(float).T.plot(kind='bar', ax=ax, color=list(KEY.values()), width=0.8, rot=0)
for cont in ax.containers:
    ax.bar_label(cont, fmt='%.3f', fontsize=8, padding=2)
ax.set(ylim=(0, 1.08), ylabel='Score', title='Test: Mô hình cũ (xám) vs. ML cải tiến (đỏ) vs. Baseline DL (xanh)')
ax.legend([f'{r} — {n}' for n, r in zip(summary.index[:3], summary['Vai trò'][:3])], loc='lower left', fontsize=9)
plt.tight_layout()
plt.show()

## 12. Phân tích lỗi của mô hình tốt nhất

In [ ]:
thr = REGISTRY[BEST_ML]['threshold']
err = R['test'][['url']].assign(label=y['test'], p=P_TEST[BEST_ML])
fn_top = err[(err['label'] == 1) & (err['p'] < thr)].sort_values('p').head(10)
fp_top = err[(err['label'] == 0) & (err['p'] >= thr)].sort_values('p', ascending=False).head(10)
print('False Negatives tự tin nhất (phishing bị chấm thấp):')
display(fn_top)
print('False Positives tự tin nhất (URL sạch bị chấm cao):')
display(fp_top)

fig, ax = plt.subplots(figsize=(8, 7))
ax.plot([0, 1], [0, 1], 'k:', label='Hiệu chỉnh hoàn hảo')
for name in [BEST_OLD, BEST_ML, BEST_DL]:
    fp_, mp_ = calibration_curve(y['test'], np.clip(P_TEST[name], 0, 1), n_bins=15, strategy='quantile')
    ax.plot(mp_, fp_, marker='o', ms=4, c=KEY[name], label=f'{name} (Brier {TEST_RESULTS[name]["Brier"]:.4f})')
ax.set(xlabel='Xác suất dự đoán', ylabel='Tỉ lệ phishing thực tế', title='Reliability diagram (test)')
ax.legend(fontsize=8.5)
plt.tight_layout()
plt.show()

## Phân tích kết quả

### A. Thứ hạng giữa các họ thuật toán (test, PR-AUC)

| Hạng | Họ | Mô hình tốt nhất | PR-AUC | F1 | Recall@FPR≤1% |
|:-:|---|---|:-:|:-:|:-:|
| 1 | **D. Ensemble** | Stacking (TF-IDF LR+NB → LightGBM) | **0.9864** | **0.9334** | **0.904** |
| 2 | B. Tuyến tính trên TF-IDF | LR + TF-IDF | 0.9786 | 0.9236 | 0.868 |
| 3 | C. Cây & Boosting (đặc trưng thủ công) | LightGBM + Optuna | 0.9264 | 0.8502 | 0.751 |
| 4 | E. Baseline DL | MLP (256-128-64) | 0.9080 | 0.8126 | 0.707 |
| 5 | A. Mô hình cũ | XGBoost tuned NB05 | 0.8772 | 0.7701 | 0.585 |

**10 vị trí đầu trên Test giữ nguyên thứ tự như trên Validation** (Stacking > Voting > LR-TFIDF > SVM > NB > LightGBM > XGBoost > CatBoost > RF > MLP). Chỉ nhóm yếu nhất (DT, LR thủ công, mô hình cũ, PR-AUC 0.84–0.88) đổi chỗ cho nhau. Chênh lệch PR-AUC giữa Validation và Test của Stacking chỉ là 0.003. Như vậy quá trình chọn mô hình trên Validation **không bị overfit**.

### B. Năm phát hiện chính (5-Whys)

1. **Biểu diễn quyết định nhiều hơn thuật toán.** Cùng là boosting, XGBoost trên 20 đặc trưng cũ đạt PR-AUC 0.877, còn LightGBM trên 52 đặc trưng mới đạt 0.926 (+0.049). Chỉ riêng một **mô hình tuyến tính** trên TF-IDF n-gram đã đạt 0.979, vượt mọi mô hình cây. *Vì sao?* Phase 3 cho thấy biểu diễn cũ để lại 21% URL mơ hồ, và không thuật toán nào vượt qua được trần lỗi đó.
2. **Phi tuyến được xác nhận lần nữa.** Trên cùng đặc trưng thủ công, LR (0.840) thua LightGBM (0.926) tới 0.086 PR-AUC, đúng như Phase 1 dự đoán. Mô hình tuyến tính chỉ thắng khi dữ liệu được chiếu lên **không gian 200k chiều** (định lý Cover).
3. **Stacking thắng vì hai nguồn tín hiệu bổ sung nhau.** SHAP beeswarm cho thấy meta-model dựa chủ yếu vào hai điểm n-gram (`nb_tfidf_score`, `lr_tfidf_score`). Đặc trưng bảo mật đóng vai trò **bộ hiệu chỉnh**: `sld_is_brand = 1` đẩy dự đoán về *an toàn*, cứu các trang chính chủ như `accounts.google.com/signin` mà n-gram `signin` dễ chấm nhầm. `tld_te` cao đẩy về phishing. `longest_token` dài (hash) cũng đẩy về phishing. Meta-model học được **khi nào tin chuỗi ký tự, khi nào tin tri thức bảo mật**, điều mà Soft Voting (trọng số cố định) không làm được (0.9811 so với 0.9864).
4. **OOF theo domain là bắt buộc.** PR-AUC OOF của LR dao động 0.81–0.97 giữa các fold, tức hiệu năng trên domain mới biến động mạnh. Nếu dùng dự đoán in-sample, meta-model sẽ tin điểm LR một cách mù quáng.
5. **Baseline DL (MLP) chỉ đứng giữa:** MLP vượt mô hình cũ (+0.031 PR-AUC) và LR trên cùng đặc trưng (+0.068), nhưng **thua mọi GBDT** trên dữ liệu bảng và thua xa Stacking (−0.078 PR-AUC, nhiều hơn 4,356 FN). Điều này khớp với hiểu biết chung: trên dữ liệu bảng cỡ vừa, GBDT vẫn vượt mạng nơ-ron truyền thẳng. MLP cũng **hiệu chỉnh kém**: ngưỡng tối ưu lên tới 0.969, do sample weight đẩy xác suất lên cao.

### C. Ý nghĩa vận hành (khoảng 100k URL test, tương đương một ngày lưu lượng)

| | Mô hình cũ | Stacking | Thay đổi |
|---|:-:|:-:|:-:|
| Phishing lọt qua (FN) | 6,518 | **1,951** | **−70.1%** |
| URL sạch bị chặn nhầm (FP) | 2,568 | **872** | **−66.0%** |
| Tỉ lệ cảnh báo sai (1 − Precision) | 14.4% | **4.2%** | Alert fatigue giảm 3.4 lần |
| Recall khi giới hạn FPR ≤ 1% | 58.5% | **90.4%** | +31.9 điểm |

- **Cải thiện đồng thời cả FP lẫn FN.** Thông thường hai loại lỗi đánh đổi lẫn nhau khi dịch ngưỡng. Việc cả hai cùng giảm hơn 66% nghĩa là **toàn bộ đường cong PR được đẩy lên**. Panel ROC phóng to cho thấy đường Stacking (đỏ) nằm trên đường mô hình cũ (xám) khoảng 30 điểm TPR ở mọi FPR ≤ 5%.
- **Khoảng tin cậy bootstrap 95% không giao nhau** giữa ba mô hình đại diện, ví dụ F1 [0.931, 0.936] so với [0.766, 0.775] của mô hình cũ. Chênh lệch có ý nghĩa thống kê.
- **Chính sách hai ngưỡng** (chọn trên Validation): vùng 🔴 *chặn tự động* bắt **85.0%** phishing với độ chính xác 98.6%, và chỉ chặn nhầm 0.34% URL sạch. Vùng 🟡 *sandbox* bắt thêm 11.7%. Tổng cộng **96.7%** phishing bị chặn hoặc gắn cờ, và chỉ 3.3% lọt qua.

### D. Phân tích lỗi: giới hạn của phương pháp chỉ dùng chuỗi URL
- **FN tự tin nhất** là các URL mà *bản thân chuỗi* trông hoàn toàn hợp lệ: bài blog trên domain bị chiếm quyền (`citrusandcandy.com/2009/11/...`), trang trên nền tảng hợp lệ (`vk.com/...`). Không có đặc trưng lexical nào phân biệt được chúng. Cần tầng enrichment như nội dung trang, tuổi domain hay danh tiếng.
- **FP tự tin nhất** có quy luật rõ: `torrentz.eu/<hash 40 ký tự>` (hash SHA-1 trông giống thư mục ngẫu nhiên của phishing kit), host là IP, và TLD `.com.br` (rủi ro cao trong dữ liệu). Cách khắc phục là thêm allowlist hoặc danh tiếng domain (ví dụ Tranco rank) làm đặc trưng.

## 13. Đóng gói pipeline inference và kiểm tra tính bất biến

Toàn bộ thành phần của mô hình tốt nhất (vectorizer, selector, preprocessor, level-0, meta-model, ngưỡng) được lưu thành **một artifact duy nhất**. Hàm `predict_urls` nhận **URL thô** và chạy đúng chuỗi xử lý của lúc train, nhờ đó không còn train/serve skew. Phép thử bất biến kiểm tra rằng **cùng một website viết theo nhiều cách** phải nhận **cùng một xác suất**.

In [ ]:
DEPLOY = 'Stacking (TF-IDF LR+NB → LightGBM)'          # pipeline được đóng gói để triển khai
preprocessor = joblib.load(TRANSFORMER_DIR / 'preprocessor.joblib')
bundle = {
    'char_vec': char_vec, 'word_vec': word_vec, 'tfidf_selector': selector, 'preprocessor': preprocessor,
    'lr_tfidf': lr_tfidf, 'cnb': cnb, 'stack_meta': stack_meta,
    'num_cols': NUM_COLS, 'tree_features': TREE_FEATURES, 'level0': LEVEL0,
    # Mọi ngưỡng đều chọn trên VALIDATION của chính mô hình được triển khai
    'thresholds': {'f1': REGISTRY[DEPLOY]['threshold'],
                   'block': threshold_for_fpr(y['val'], REGISTRY[DEPLOY]['p_val'], 0.003),
                   'review': threshold_for_fpr(y['val'], REGISTRY[DEPLOY]['p_val'], 0.03)},
    'deployed_model': DEPLOY, 'best_model_on_validation': BEST_ML, 'old_xgb': old_xgb,
}
joblib.dump(bundle, MODEL_DIR / 'phishing_detector.joblib', compress=3)
print(f'Đã lưu artifacts/models/phishing_detector.joblib '
      f'({(MODEL_DIR / "phishing_detector.joblib").stat().st_size / 1e6:.1f} MB)')


def predict_urls(urls, b=bundle) -> pd.DataFrame:
    """URL thô → xác suất phishing của Stacking + mô hình cũ (đối chiếu)."""
    urls = list(urls)
    Fn = build_feature_frame(urls)
    tld_te = b['preprocessor'].transform(Fn[b['num_cols'] + ['tld']])['tld'].to_numpy(np.float32)
    Xt = Fn[b['num_cols']].astype(np.float32).assign(tld_te=tld_te)[b['tree_features']]
    norm = [normalize_url(u) for u in urls]
    Tm = b['tfidf_selector'].transform(sparse.hstack([b['char_vec'].transform(norm), b['word_vec'].transform(norm)]).tocsr())
    Xm = Xt.assign(lr_tfidf_score=b['lr_tfidf'].predict_proba(Tm)[:, 1].astype(np.float32),
                   nb_tfidf_score=b['cnb'].predict_proba(Tm)[:, 1].astype(np.float32))
    p = b['stack_meta'].predict_proba(Xm)[:, 1]
    verdict = np.select([p >= b['thresholds']['block'], p >= b['thresholds']['review']], ['🔴 BLOCK', '🟡 REVIEW'], '🟢 ALLOW')
    return pd.DataFrame({'URL': urls, 'P(phishing) Stacking': p, 'Quyết định': verdict,
                         'P(phishing) mô hình cũ': b['old_xgb'].predict_proba(legacy_feature_frame(urls))[:, 1]})


# Kiểm tra tính nhất quán: pipeline inference phải tái tạo đúng xác suất đã tính trên test
chk = predict_urls(R['test']['url'].iloc[:2000])
print(f'Sai khác tối đa giữa predict_urls và P_TEST: '
      f'{np.abs(chk["P(phishing) Stacking"].to_numpy() - P_TEST[DEPLOY][:2000]).max():.2e}')

variants = {'google': ['google.com', 'https://google.com', 'www.google.com', 'HTTPS://WWW.Google.com/'],
            'github': ['github.com/scikit-learn/scikit-learn', 'https://github.com/scikit-learn/scikit-learn/'],
            'paypal-phish': ['paypal.com.secure-login.verify-account.xyz/webscr.php?cmd=_login',
                             'http://paypal.com.secure-login.verify-account.xyz/webscr.php?cmd=_login']}
extra = ['accounts.google.com/signin', 'paypa1.com/signin', 'http://192.168.10.5/login/paypal/index.php',
         'ikmbtfomlqp.deliverdarken.ru/start_page.exe', 'en.wikipedia.org/wiki/Phishing']
demo = predict_urls([u for us in variants.values() for u in us] + extra)
display(demo.style.format({'P(phishing) Stacking': '{:.3f}', 'P(phishing) mô hình cũ': '{:.3f}'})
        .background_gradient(subset=['P(phishing) Stacking', 'P(phishing) mô hình cũ'], cmap='RdYlGn_r', vmin=0, vmax=1))
spread = pd.DataFrame({g: demo.set_index('URL').loc[us, ['P(phishing) Stacking', 'P(phishing) mô hình cũ']]
                       .agg(lambda s: s.max() - s.min()) for g, us in variants.items()}).T
print('Độ lệch xác suất (max − min) giữa các cách viết của cùng một URL (càng gần 0 càng tốt):')
display(spread.round(4))

## 14. Kết luận

- **Pipeline inference nhất quán:** `predict_urls()` tái tạo **chính xác** xác suất trên test (sai khác 0.0). Mọi cách viết của cùng một URL (có hay không scheme, `www.`, chữ hoa, `/` cuối) nhận **cùng một xác suất** (độ lệch 0.000), trong khi mô hình cũ dao động tới 0.23 với `github.com`. Train/serve skew đã được loại bỏ.
- **Mô hình đề xuất triển khai:** **Stacking (TF-IDF LR + Complement NB → LightGBM)**. Trên **domain chưa từng thấy** mô hình đạt PR-AUC **0.986**, ROC-AUC **0.995**, F1 **0.933** và Recall **90.4%** tại FPR 1%. So với mô hình cũ, số phishing lọt qua giảm 70% và số lần chặn nhầm giảm 66%. Mô hình huấn luyện trên CPU, artifact khoảng 19 MB, và inference chỉ gồm thao tác chuỗi, một phép nhân ma trận thưa và vài trăm cây quyết định.
- **Mọi lựa chọn đều có bằng chứng:** phi tuyến (Phase 1) → làm sạch và chia theo domain (Phase 2) → đặc trưng có cơ sở bảo mật và chọn lọc thống kê (Phase 3) → tuning trên Validation và đánh giá Test đúng một lần (Phase 4).
- **Hướng tiếp theo:** (1) thêm tầng enrichment (danh tiếng domain, RDAP/DNS) cho vùng 🟡 để xử lý FN kiểu domain bị chiếm quyền; (2) hiệu chỉnh xác suất (isotonic) trên một tập calibration riêng; (3) đánh giá theo **temporal split** với feed mới (PhishTank, OpenPhish, URLhaus), vì dataset hiện tại có artefact thời gian.